# 17. Avaliação da calibração da API

A política de resposta é mantida fixa. A referência usa os pontos municipais com chave utilizável, sem estratos operacionais. Todas as chaves das amostras já consultadas ficam fora da nova avaliação. Calibração e teste são separados por blocos definidos pelo centro de cada chave, preservando todos os seus pontos na mesma partição. A avaliação mede discrepância em relação ao cadastro municipal, sem validação de campo da identidade ou da posição. As saídas são gravadas em um diretório exclusivo de execução.


In [1]:
from pathlib import Path
import sys, json, time, hashlib, os
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import geopandas as gpd
import httpx
import matplotlib.pyplot as plt
from pyproj import Transformer
from sklearn.metrics import roc_auc_score
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/config.py').exists())
sys.path.insert(0, str(ROOT))
from src import config
from src.metrics import chave_identidade
from src.api.calibracao import Calibrador, carregar
P = config.PROCESSED_DIR
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SAIDA = config.OUTPUT_DIR/'dissertacao'/'api'/RUN_ID
T, F = SAIDA/'tables', SAIDA/'figures'
for d in [T,F]: d.mkdir(parents=True, exist_ok=False)
RANDOM_STATE = 42
API = os.environ.get('API_AVALIACAO_URL','http://localhost:58001')
BLOCO_M = 2000
N_POR_METADE = 20000
print('Saida da execucao:', SAIDA)

Saida da execucao: C:\Users\mateu\OneDrive\Documentos\UFMG\Mestrado\geocoding-quality-analysis\outputs\dissertacao\api\20261028T135932066950Z


## 1. População e separação das amostras

Cada ponto com chave utilizável constitui uma unidade amostral. Chaves presentes no desenvolvimento ou em avaliações anteriores são excluídas antes do sorteio. A mediana das coordenadas municipais de cada chave define o seu bloco de 2 km. Todos os pontos de uma chave recebem o mesmo bloco, inclusive quando suas localizações ocupam células diferentes da grade. Os blocos de chaves são sorteados entre calibração e teste. Essa regra impede repetição de chaves entre as duas partes, mas não cria uma faixa de afastamento espacial. As amostras anteriores abrangem a cidade, de modo que a separação em relação ao desenvolvimento é por chave e não por território.


In [2]:
bh_total = gpd.read_parquet(P/'bhmap_bh.parquet')
bh = bh_total.copy()
assert len(bh) == 757113
assert bh.index.is_unique
log_b, num_b = chave_identidade(bh.SIGLA_TIPO.fillna('')+' '+bh.NOME_LOGRA.fillna(''), bh.NUMERO_IMO)
bh['chave'] = (log_b.astype('string')+'|'+num_b.astype('string')).where(log_b.notna() & log_b.ne('') & num_b.notna()).to_numpy()
dev_path = ROOT/'scratch/baseline/input_bh.csv'
dev = pd.read_csv(dev_path)
log_d, num_d = chave_identidade(dev.logradouro, dev.numero)
chaves_dev = set((log_d.astype('string')+'|'+num_d.astype('string')).dropna())
old_path = config.TAB_DIR/'nb17_medicoes.csv'
old = pd.read_csv(old_path)
assert old.idx.isin(bh.index).all()
chaves_anteriores = set(bh.loc[old.idx,'chave'].dropna())
excluidas = chaves_dev | chaves_anteriores
pop = bh[bh.chave.notna() & ~bh.chave.isin(excluidas)].copy()
centros = pd.DataFrame({'chave':pop.chave,'x':pop.geometry.x,'y':pop.geometry.y}).groupby('chave')[['x','y']].transform('median')
pop['bloco'] = (centros.x//BLOCO_M).astype(int).astype(str)+'_'+(centros.y//BLOCO_M).astype(int).astype(str)
blocos = np.array(sorted(pop.bloco.unique()))
rng = np.random.default_rng(RANDOM_STATE)
calib_blocos = set(rng.permutation(blocos)[:len(blocos)//2])
pop['metade'] = np.where(pop.bloco.isin(calib_blocos),'calibracao','teste')
amostra = pd.concat([g.sample(N_POR_METADE,random_state=RANDOM_STATE) for _,g in pop.groupby('metade')])
assert not amostra.chave.isin(excluidas).any()
assert amostra.groupby('chave').metade.nunique().max() == 1
assert amostra.groupby('bloco').metade.nunique().max() == 1
assert set(amostra.loc[amostra.metade.eq('calibracao'),'chave']).isdisjoint(set(amostra.loc[amostra.metade.eq('teste'),'chave']))
colunas_amostra = ['chave','bloco','metade','SIGLA_TIPO','NOME_LOGRA','NUMERO_IMO','NOME_BAIRR','geometry']
amostra[colunas_amostra].to_parquet(T/'amostra_congelada.parquet')
desenho = pd.DataFrame([
    {'etapa':'pontos municipais completos','unidade':'pontos','n':len(bh)},
    {'etapa':'com chave utilizavel','unidade':'pontos','n':int(bh.chave.notna().sum())},
    {'etapa':'desenvolvimento','unidade':'chaves excluidas','n':len(chaves_dev)},
    {'etapa':'avaliacoes anteriores','unidade':'chaves excluidas','n':len(chaves_anteriores)},
    {'etapa':'uniao das exclusoes','unidade':'chaves excluidas','n':len(excluidas)},
    {'etapa':'populacao elegivel','unidade':'pontos','n':len(pop)},
    {'etapa':'populacao elegivel','unidade':'chaves','n':pop.chave.nunique()},
    *[{'etapa':m,'unidade':'pontos amostrados','n':len(g)} for m,g in amostra.groupby('metade')],
    *[{'etapa':m,'unidade':'chaves amostradas','n':g.chave.nunique()} for m,g in amostra.groupby('metade')],
    *[{'etapa':m,'unidade':'blocos da populacao','n':g.bloco.nunique()} for m,g in pop.groupby('metade')]])
desenho.to_csv(T/'nb17_desenho_amostral.csv',index=False)
print(desenho.to_string(index=False))
sha = lambda f: hashlib.sha256(Path(f).read_bytes()).hexdigest()
esperado = json.loads((config.TAB_DIR/'nb17_manifesto.json').read_text(encoding='utf-8'))
congelamento = {'congelado_utc':datetime.now(timezone.utc).isoformat(),'semente':RANDOM_STATE,'bloco_m':BLOCO_M,'n_por_metade':N_POR_METADE,'unidade_amostral':'ponto municipal','unidade_particao':'bloco do centro mediano da chave','buffer_espacial':False,'parametros_alterados_apos_teste':False,'versao_politica_esperada':esperado['versao_politica'],'versao_dados_esperada':esperado['versao_dados'],'amostra_sha256':sha(T/'amostra_congelada.parquet'),'entradas':{str(f.relative_to(ROOT)):sha(f) for f in [P/'bhmap_bh.parquet',dev_path,old_path]},'codigo_api':{str(f.relative_to(ROOT)):sha(f) for f in sorted((ROOT/'src/api').glob('*.py'))},'instalar_tabela':False}
(T/'congelamento.json').write_text(json.dumps(congelamento,ensure_ascii=False,indent=2),encoding='utf-8')


                      etapa             unidade      n
pontos municipais completos              pontos 757113
       com chave utilizavel              pontos 757113
            desenvolvimento    chaves excluidas  19798
      avaliacoes anteriores    chaves excluidas  39325
        uniao das exclusoes    chaves excluidas  59123
         populacao elegivel              pontos 665423
         populacao elegivel              chaves 555527
                 calibracao   pontos amostrados  20000
                      teste   pontos amostrados  20000
                 calibracao   chaves amostradas  19743
                      teste   chaves amostradas  19713
                 calibracao blocos da populacao     55
                      teste blocos da populacao     56
Out[2]: 1551


## 2. Consultas à política congelada

Antes de consultar as amostras, o serviço deve apresentar as versões esperadas em uma consulta da amostra de desenvolvimento. As versões devem permanecer iguais em todas as respostas, e os arquivos locais do serviço devem manter seus hashes. Esses controles documentam a execução, mas não substituem a identificação do processo e da carga efetivamente servidos. Cada consulta usa tipo, nome, número e bairro, retendo o primeiro candidato. A latência é medida individualmente e a vazão é calculada separadamente.


In [ ]:
saude = httpx.get(API+'/health',timeout=10).json()
assert saude['status'] == 'ok' and saude['enderecos'] == 590615
cliente = httpx.Client(base_url=API, timeout=30, limits=httpx.Limits(max_connections=16))
def consultar(linha):
    texto = f"{linha.SIGLA_TIPO} {linha.NOME_LOGRA}, {int(linha.NUMERO_IMO)}, {linha.NOME_BAIRR}"
    t0 = time.perf_counter()
    r = cliente.post('/geocode', json={'endereco': texto, 'municipio': 'Belo Horizonte', 'limite': 1})
    ms = (time.perf_counter()-t0)*1000
    base = {'idx': linha.Index, 'status_http': r.status_code, 'latencia_ms': ms}
    corpo = r.json() if r.status_code == 200 else {}
    if not corpo.get('candidatos'):
        return {**base, 'nivel': 'sem_resposta', 'confianca': 'sem_resposta'}
    m = corpo['candidatos'][0]; q = m['qualidade']
    return {**base, 'lat': m['latitude'], 'lon': m['longitude'], 'nivel': m['nivel_precisao'],
            'confianca': q['confianca_posicional'], 'identidade': q['incerteza_identidade'],
            'similaridade': m['similaridade_logradouro'], 'versao_dados': m['versao_dados'],
            'versao_politica': q['calibracao']['versao_politica'], 'status_calibracao': q['calibracao']['status']}
d = dev.iloc[0]
pre = cliente.post('/geocode',json={'endereco':f'{d.logradouro}, {d.numero}','municipio':'Belo Horizonte','limite':1})
pre.raise_for_status()
pre_c = pre.json()['candidatos'][0]
assert pre_c['versao_dados'] == congelamento['versao_dados_esperada']
assert pre_c['qualidade']['calibracao']['versao_politica'] == congelamento['versao_politica_esperada']
inicio = time.perf_counter()
with ThreadPoolExecutor(max_workers=8) as ex:
    resp = pd.DataFrame(list(ex.map(consultar, amostra.itertuples())))
decorrido = time.perf_counter()-inicio
resp = resp.set_index('idx').join(amostra[['metade','bloco','chave']].assign(xr=amostra.geometry.x, yr=amostra.geometry.y))
para_utm = Transformer.from_crs('EPSG:4326', bh.crs, always_xy=True)
ok = resp.lat.notna()
x, y = para_utm.transform(resp.loc[ok,'lon'].to_numpy(), resp.loc[ok,'lat'].to_numpy())
resp.loc[ok,'erro_m'] = np.hypot(x-resp.loc[ok,'xr'].to_numpy(), y-resp.loc[ok,'yr'].to_numpy())
politicas, versoes = resp.versao_politica.dropna().unique(), resp.versao_dados.dropna().unique()
assert len(politicas) == 1 and len(versoes) == 1, (politicas, versoes)
assert resp.status_http.isin([200, 404]).all(), resp.status_http.value_counts()
VERSAO_POLITICA, VERSAO_DADOS = politicas[0], versoes[0]
assert VERSAO_POLITICA == congelamento['versao_politica_esperada']
assert VERSAO_DADOS == congelamento['versao_dados_esperada']
assert all(sha(ROOT/f)==h for f,h in congelamento['codigo_api'].items())
latencia = {'consultas':len(resp),'segundos':decorrido,'consultas_por_segundo':len(resp)/decorrido,'latencia_mediana_ms':resp.latencia_ms.median(),'latencia_p90_ms':resp.latencia_ms.quantile(.9),'latencia_p95_ms':resp.latencia_ms.quantile(.95),'concorrencia':8}
(T/'latencia.json').write_text(json.dumps(latencia,indent=2),encoding='utf-8')
print(latencia)
print('HTTP', resp.status_http.value_counts().to_dict(), '| status de calibracao', resp.status_calibracao.value_counts(dropna=False).to_dict())
print('versao_politica', VERSAO_POLITICA); print('versao_dados', VERSAO_DADOS)
resp.drop(columns=['lat','lon','xr','yr']).to_csv(T/'nb17_medicoes.csv')

## 3. Cobertura dos quantis condicionais

A tabela é estimada somente na calibração. Cada resposta de teste recebe a mediana e o P90 de sua célula de nível e confiança. Quando a célula tem menos de cem casos, usa-se a célula do nível ou a global. A cobertura agregada conta quantas discrepâncias ficam abaixo do quantil atribuído à respectiva resposta. Ela não equivale a aplicar um único P90 global a todas as respostas. Os intervalos usam mil reamostragens dos blocos do teste, preservando a multiplicidade dos blocos sorteados. São condicionais à tabela estimada, sem reamostrar a etapa de calibração.


In [ ]:
def estatisticas(e):
    e = np.asarray(e, float)
    return {'n': len(e), 'erro_mediano_m': float(np.median(e)), 'erro_ce90_m': float(np.quantile(e, .9)),
            'acima_30_pct': 100*float((e > 30).mean()), 'acima_100_pct': 100*float((e > 100).mean())}
medido = resp[resp.erro_m.notna()]
cal, tes = medido[medido.metade == 'calibracao'], medido[medido.metade == 'teste']
linhas = [{'nivel': '', 'confianca': '', 'base': 'global', **estatisticas(cal.erro_m)}]
for nv, g in cal.groupby('nivel'):
    linhas.append({'nivel': nv, 'confianca': '', 'base': 'nivel', **estatisticas(g.erro_m)})
for (nv, cf), g in cal.groupby(['nivel','confianca']):
    linhas.append({'nivel': nv, 'confianca': cf, 'base': 'nivel+confianca', **estatisticas(g.erro_m)})
tabela = pd.DataFrame(linhas)
tabela = tabela[tabela.n >= 100].reset_index(drop=True)
def celula(nv, cf):
    for k in [(nv, cf), (nv, ''), ('', '')]:
        l = tabela[(tabela.nivel == k[0]) & (tabela.confianca == k[1])]
        if len(l): return l.iloc[0]
tes = tes.copy()
cel = [celula(nv, cf) for nv, cf in zip(tes.nivel, tes.confianca)]
tes['med_cal'] = [c.erro_mediano_m for c in cel]; tes['ce90_cal'] = [c.erro_ce90_m for c in cel]
tes['base_usada'] = [c.base for c in cel]
tes['abaixo_med'] = tes.erro_m <= tes.med_cal; tes['abaixo_ce90'] = tes.erro_m <= tes.ce90_cal
tes = tes.reset_index()
posicoes = [np.asarray(v) for v in tes.groupby('bloco').indices.values()]
rng = np.random.default_rng(RANDOM_STATE)
def ic_bloco(func, reps=1000):
    vals = []
    for _ in range(reps):
        b = rng.integers(len(posicoes), size=len(posicoes))
        vals.append(func(tes.iloc[np.concatenate([posicoes[k] for k in b])]))
    return np.percentile(vals, [2.5, 97.5])
cobertura = []
for nome, g in [('global', tes)] + [(nv, g) for nv, g in tes.groupby('nivel')]:
    filtro = (lambda s: s) if nome == 'global' else (lambda s, nv=nome: s[s.nivel == nv])
    ic_med = ic_bloco(lambda s: filtro(s).abaixo_med.mean()*100)
    ic_ce = ic_bloco(lambda s: filtro(s).abaixo_ce90.mean()*100)
    cobertura.append({'grupo': nome, 'n_teste': len(g), 'cobertura_mediana_pct': 100*g.abaixo_med.mean(), 'ic_mediana': f'{ic_med[0]:.1f} a {ic_med[1]:.1f}',
                      'cobertura_ce90_pct': 100*g.abaixo_ce90.mean(), 'ic_ce90': f'{ic_ce[0]:.1f} a {ic_ce[1]:.1f}',
                      **{f'teste_{k}': v for k, v in estatisticas(g.erro_m).items() if k != 'n'}})
cobertura = pd.DataFrame(cobertura)
cobertura.to_csv(T/'nb17_cobertura_teste.csv', index=False)
print('Tabela estimada na calibracao'); print(tabela.round(2).to_string(index=False))
print('\nCobertura na metade de teste'); print(cobertura.round(2).to_string(index=False))

Os resultados desta seção descrevem a população e as respostas da execução identificada no manifesto. Proximidade ao nível nominal, isoladamente, não demonstra equivalência, acurácia de campo ou validade em outro município.


## 4. Recusas e discriminação

As recusas pertencem ao denominador das consultas, mas não ao cálculo de discrepância entre respostas. A discriminação de discrepâncias acima de 30 m é calculada para nível e classe de confiança. A diferença entre as AUCs usa os mesmos blocos reamostrados, sem transformar ausência de diferença detectável em prova de equivalência. Os resultados do desenvolvimento não constituem uma população comparável para medir o efeito de reutilização da amostra.


In [ ]:
ordem_nivel = {'numero_exato': 0, 'numero_aproximado': 1, 'logradouro': 2}
ordem_conf = {'alta': 0, 'moderada': 1, 'baixa': 2}
resposta = resp.groupby('metade').apply(lambda g: pd.Series({
    'consultas': len(g), 'respondidas_pct': 100*g.nivel.ne('sem_resposta').mean(),
    **{f'{k}_pct': 100*g.nivel.eq(k).mean() for k in ['numero_exato','numero_aproximado','logradouro','sem_resposta']}}))
resposta.to_csv(T/'nb17_taxa_resposta.csv')
print(resposta.round(2).to_string())
def auc(s, col, ordem):
    return roc_auc_score(s.erro_m > 30, s[col].map(ordem))
disc = []
for col, ordem in [('nivel', ordem_nivel), ('confianca', ordem_conf)]:
    ic = ic_bloco(lambda s: auc(s, col, ordem), reps=500)
    disc.append({'sinal': col, 'auc_teste': auc(tes, col, ordem), 'ic95': f'{ic[0]:.3f} a {ic[1]:.3f}', 'auc_calibracao': auc(cal, col, ordem)})
disc = pd.DataFrame(disc); disc.to_csv(T/'nb17_discriminacao.csv', index=False)
print(disc.round(3).to_string(index=False))

delta_obs = auc(tes,'confianca',ordem_conf)-auc(tes,'nivel',ordem_nivel)
ic_delta = ic_bloco(lambda s: auc(s,'confianca',ordem_conf)-auc(s,'nivel',ordem_nivel),reps=500)
(T/'delta_auc_pareada.json').write_text(json.dumps({'delta_auc_confianca_menos_nivel':delta_obs,'ic95':ic_delta.tolist(),'repeticoes':500},indent=2),encoding='utf-8')
print('Diferenca pareada de AUC:',delta_obs,ic_delta)


In [ ]:
cv = cobertura.set_index('grupo')
fig, ax = plt.subplots(figsize=(8,4.5))
ypos = np.arange(len(cv))
ax.scatter(cv.cobertura_ce90_pct,ypos,color='#367EA1')
ax.axvline(90,color='#D04A38',linestyle='--')
ax.set_yticks(ypos,cv.index)
ax.set(xlabel='Discrepâncias abaixo do P90 condicional (%)',title='Cobertura na amostra de teste',xlim=(0,100))
for j,(v,ic) in enumerate(zip(cv.cobertura_ce90_pct,cv.ic_ce90)):
    ax.text(2,j,f'{v:.1f}% (IC {ic})'.replace('.',','),va='center',fontsize=9)
fig.tight_layout()
fig.savefig(F/'nb17_cobertura_condicional.png',dpi=200,bbox_inches='tight')
plt.show()


As AUCs e a diferença pareada descrevem a discriminação nesta amostra. A cobertura condicional deve ser lida com seus denominadores, recusas, intervalos e a regra de recuo das células pequenas.


## 5. Tabela candidata

O validador da API verifica a estrutura da tabela, os quantis, as taxas e a compatibilidade de versões. Essa aceitação técnica não comprova a validade científica da calibração. A tabela candidata permanece no diretório desta execução e não substitui a tabela distribuída nem é instalada no serviço.


In [ ]:
cand = tabela.copy()
cand.insert(0, 'codigo_municipio', '3106200')
cand['n'] = cand.n.astype(int)
for c in ['erro_mediano_m','erro_ce90_m','acima_30_pct','acima_100_pct']: cand[c] = cand[c].round(2)
cand['versao_dados'] = VERSAO_DADOS; cand['versao_politica'] = VERSAO_POLITICA
cand['tipo_consulta'] = 'geocode'; cand['amostra'] = 'teste_independente'
cand['fonte_referencia'] = 'BHMap completo, chaves separadas por blocos de 2 km, amostra de calibracao'
destino = T/'nb17_calibracao_candidata.csv'
cand.to_csv(destino, index=False)
calibrador = carregar(destino)
print('municipios aceitos pelo validador da API:', calibrador.municipios_calibrados)
assert calibrador.municipios_calibrados == ['3106200']
from src.api.schemas import NivelPrecisao, ClasseConfianca
print(calibrador.consultar('3106200', NivelPrecisao('numero_exato'), ClasseConfianca('alta'), versao_dados=VERSAO_DADOS, versao_politica=VERSAO_POLITICA))
manifesto = {'concluido_utc': datetime.now(timezone.utc).isoformat(), 'notebook': '17_teste_operacional_independente.ipynb', 'api': API, 'saude': saude,
             'versao_politica': VERSAO_POLITICA, 'versao_dados': VERSAO_DADOS, 'bloco_m': BLOCO_M, 'n_por_metade': N_POR_METADE,
             'semente': RANDOM_STATE, 'segundos': round(decorrido, 1), 'tabela_instalada_na_imagem': False,
             'bhmap_sha256': hashlib.sha256((P/'bhmap_bh.parquet').read_bytes()).hexdigest(), 'populacao': 'pontos municipais com chave utilizavel, sem chaves de amostras anteriores', 'run_id': RUN_ID, 'congelamento_sha256': sha(T/'congelamento.json')}
(T/'nb17_manifesto.json').write_text(json.dumps(manifesto, ensure_ascii=False, indent=2), encoding='utf-8')
print(cand.to_string(index=False))

## Conclusão

A avaliação separa a estimação dos quantis e a verificação de sua cobertura, mantendo a política fixa e as chaves fora das amostras anteriores. Sua interpretação depende das saídas da execução completa e dos controles de proveniência. O desenho não elimina dependência entre blocos vizinhos, não confirma identidade física e não demonstra validade externa. O [NB18](18_restricao_territorial.ipynb) examina as restrições territoriais do pareamento cadastral.
